# 03 — Baseline CNN

Train a small convolutional network from scratch for binary real-vs-AI image classification. The standard experiment uses the fixed train, validation, and test splits created in notebook 01.

## Setup

Load the reusable data, model, training, and evaluation utilities. The test split remains unused until the best epoch has been selected using validation loss.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.optim import AdamW


def find_repo_root(start: Path) -> Path:
    """Find the repository root from the current working directory."""
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "experiments.yaml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT))

from src.data.config import load_config
from src.data.loaders import create_dataloaders
from src.evaluation import evaluate_predictions, save_evaluation_outputs
from src.models import BaselineCNN, count_trainable_parameters
from src.training import fit, set_seed

In [ ]:
config: dict = load_config()
SEED: int = config["seed"]
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

MANIFEST_PATH = REPO_ROOT / "data" / "dataset.csv"
PROCESSED_ROOT = REPO_ROOT / "data" / "processed"
CHECKPOINT_PATH = REPO_ROOT / "results" / "checkpoints" / "baseline_cnn.pt"
METRICS_ROOT = REPO_ROOT / "results" / "metrics" / "baseline_cnn"
FIGURES_ROOT = REPO_ROOT / "results" / "figures" / "baseline_cnn"
METRICS_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)

## DataLoaders

Use deterministic transforms for the first baseline. Training is shuffled with the project seed, while validation and test retain stable order.

In [ ]:
loader_config = config["dataloader"]
dataloaders = create_dataloaders(
    manifest=MANIFEST_PATH,
    processed_root=PROCESSED_ROOT,
    batch_size=loader_config["batch_size"],
    num_workers=loader_config["num_workers"],
    seed=SEED,
    pin_memory=device.type == "cuda",
)

print("Dataset sizes:", {split: len(loader.dataset) for split, loader in dataloaders.items()})
sample_batch = next(iter(dataloaders["train"]))
print("Image batch shape:", tuple(sample_batch["image"].shape))
print("Label batch shape:", tuple(sample_batch["label"].shape))
print("Pixel range:", float(sample_batch["image"].min()), float(sample_batch["image"].max()))

## Model

The baseline has four Conv–BatchNorm–ReLU–MaxPool blocks, global average pooling, dropout, and one output logit. `BCEWithLogitsLoss` applies the sigmoid operation internally.

In [ ]:
model_config = config["baseline_cnn"]
model = BaselineCNN(
    channels=tuple(model_config["channels"]),
    dropout=model_config["dropout"],
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(
    model.parameters(),
    lr=model_config["learning_rate"],
    weight_decay=model_config["weight_decay"],
)

with torch.inference_mode():
    sample_logits = model(sample_batch["image"].to(device))

print(f"Trainable parameters: {count_trainable_parameters(model):,}")
print("Logit shape:", tuple(sample_logits.shape))
assert sample_logits.shape == sample_batch["label"].shape

## Train and select the best epoch

Only train and validation data are used here. Early stopping monitors validation loss and restores the best checkpoint when training finishes.

In [ ]:
fit_result = fit(
    model=model,
    train_dataloader=dataloaders["train"],
    val_dataloader=dataloaders["val"],
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=model_config["epochs"],
    patience=model_config["early_stopping_patience"],
    min_delta=model_config["early_stopping_min_delta"],
    checkpoint_path=CHECKPOINT_PATH,
)

history = pd.DataFrame(fit_result.history)
history.to_csv(METRICS_ROOT / "training_history.csv", index=False)
print(f"Best epoch: {fit_result.best_epoch}")
print(f"Best validation loss: {fit_result.best_val_loss:.4f}")
history

## Learning curves

Compare train and validation loss and F1 to identify convergence and overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["epoch"], history["train_loss"], marker="o", label="train")
axes[0].plot(history["epoch"], history["val_loss"], marker="o", label="validation")
axes[0].axvline(fit_result.best_epoch, color="black", linestyle="--", label="best epoch")
axes[0].set_title("Loss by epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE loss")
axes[0].legend()

axes[1].plot(history["epoch"], history["train_f1"], marker="o", label="train")
axes[1].plot(history["epoch"], history["val_f1"], marker="o", label="validation")
axes[1].axvline(fit_result.best_epoch, color="black", linestyle="--", label="best epoch")
axes[1].set_title("F1 by epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1")
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGURES_ROOT / "training_curves.png", dpi=150)
plt.show()

## Final test evaluation

Evaluate the restored best checkpoint once on the previously untouched test split. Save overall metrics, individual predictions, per-generator metrics, the confusion matrix, and the ROC curve.

In [ ]:
test_result = evaluate_predictions(
    model=model,
    dataloader=dataloaders["test"],
    criterion=criterion,
    device=device,
    threshold=0.5,
)
saved_paths = save_evaluation_outputs(
    test_result, output_dir=METRICS_ROOT, figures_dir=FIGURES_ROOT
)

print("Test metrics:")
display(pd.Series(test_result.metrics.to_dict(), name="value"))
print("Per-generator metrics:")
display(test_result.per_generator)
print("Saved outputs:")
for name, path in saved_paths.items():
    print(f"  {name}: {path.relative_to(REPO_ROOT)}")